In [1]:
import os
import io
import copy
import random
from tqdm.auto import tqdm 

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)
train_dataset = train_dataset_full
test_dataset = test_dataset_full

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

Files already downloaded and verified
Files already downloaded and verified
train - test dataset ready.


In [3]:
#Simple Tanhh Best Task-Aware α: 0.410, Seed: 43.0

In [4]:
def load_model(path="full_model.pth"):
    print("\nLoading pretrained ResNet18 model...")
    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Modify final layer for 2 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 10)

    # Freeze all layers except layer3, layer4, fc
    for param in model.parameters():
        param.requires_grad = False
    for layer in [model.layer3, model.layer4, model.fc]:
        for param in layer.parameters():
            param.requires_grad = True

    # Load fine-tuned weights if available
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded fine-tuned weights from '{path}'")
    else:
        print(f"Warning: '{path}' not found. Model is not fine-tuned yet.")

    return model.to(device)


model = load_model()
model.eval()



Loading pretrained ResNet18 model...


/tmp/ipykernel_8763/1370779204.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=device))


Loaded fine-tuned weights from 'full_model.pth'


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
class Encoder(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.features = nn.Sequential(
            trained_model.conv1,
            trained_model.bn1,
            trained_model.relu,
            trained_model.maxpool,
            trained_model.layer1,
            trained_model.layer2,
            trained_model.layer3
        )

    def forward(self, x):
        with torch.no_grad():
            return self.features(x)


encoder = Encoder(model).to(device)
encoder.eval()


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decode = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.decode(x)


In [6]:
class ReconstructedCIFAR10(Dataset):
    def __init__(self, original_dataset, reconstructed_images):
        assert len(original_dataset) == len(reconstructed_images)
        self.dataset = original_dataset
        self.recons = reconstructed_images

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img = self.recons[idx]
        label = self.dataset[idx][1]
        return img, label

best_alpha, best_seed = 0.41, 43
decoder_path = f"STanh_TargetAlpha/decoder{best_alpha:.2f}_seed{best_seed}.pth"

decoder = Decoder().to(device)
decoder.load_state_dict(torch.load(decoder_path, map_location=device))
decoder.eval()
print(f"Loaded decoder: {decoder_path}")

def reconstruct_dataset(dataset, encoder, decoder, batch_size=64):
    reconstructed_images = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="Reconstructing dataset"):
            imgs = imgs.to(device)
            latents = encoder(imgs)
            recons = decoder(latents)
            reconstructed_images.append(recons.cpu())
    reconstructed_images = torch.cat(reconstructed_images)
    return reconstructed_images


recon_train_imgs = reconstruct_dataset(train_dataset_full, encoder, decoder)
recon_test_imgs = reconstruct_dataset(test_dataset_full, encoder, decoder)

train_dataset_recon = ReconstructedCIFAR10(train_dataset_full, recon_train_imgs)
test_dataset_recon = ReconstructedCIFAR10(test_dataset_full, recon_test_imgs)

train_loader_recon = DataLoader(train_dataset_recon, batch_size=64, shuffle=True)
test_loader_recon = DataLoader(test_dataset_recon, batch_size=64, shuffle=False)
test_loader_orig = DataLoader(test_dataset_full, batch_size=64, shuffle=False)

print("Reconstructed datasets ready!")


Loaded decoder: STanh_TargetAlpha/decoder0.41_seed43.pth


/tmp/ipykernel_8763/327286791.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder.load_state_dict(torch.load(decoder_path, map_location=device))


Reconstructing dataset:   0%|          | 0/782 [00:00<?, ?it/s]

Reconstructing dataset:   0%|          | 0/157 [00:00<?, ?it/s]

Reconstructed datasets ready!


In [7]:
num_classes = 10
epochs = 20
lr = 1e-4
patience = 4

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

for param in model.parameters():
    param.requires_grad = False
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
counter = 0

for epoch in range(epochs):
    model.train()
    running_loss, running_corrects, total = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader_recon, desc=f"Training Epoch [{epoch+1}/{epochs}]", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * imgs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = running_corrects.double() / total

    model.eval()
    val_loss, val_corrects, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(test_loader_recon, desc="Validating", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * imgs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_corrects.double() / val_total

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

model.load_state_dict(best_model_wts)

print("Training info: Dataset = Reconstructed (train_loader_recon), Classifier = Frozen layers (follows original classifier arch)")

def evaluate_df(model, loaders_dict):
    results = {"Dataset": [], "CrossEntropy": [], "Accuracy": []}
    model.eval()
    with torch.no_grad():
        for name, loader in loaders_dict.items():
            total, correct = 0, 0
            total_loss = 0.0
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                total_loss += loss.item() * imgs.size(0)
                _, preds = torch.max(outputs, 1)
                correct += torch.sum(preds == labels.data)
                total += labels.size(0)
            results["Dataset"].append(name)
            results["CrossEntropy"].append(total_loss / total)
            results["Accuracy"].append((correct.double() / total).item())

    return pd.DataFrame(results)

loaders_to_eval = {
    "Reconstructed Test Set": test_loader_recon,
    "Original Test Set": test_loader_orig
}

eval_df = evaluate_df(model, loaders_to_eval)
print(eval_df)


Training Epoch [1/20]:   0%|          | 0/782 [00:00<?, ?it/s]

Validating:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 0.1003 | Train Acc: 0.9711
Val   Loss: 1.4168 | Val   Acc: 0.7773


Training Epoch [2/20]:   0%|          | 0/782 [00:00<?, ?it/s]

Validating:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 0.0304 | Train Acc: 0.9911
Val   Loss: 1.5568 | Val   Acc: 0.7800


... (35 outputs left out)

Train Loss: 0.0006 | Train Acc: 0.9998
Val   Loss: 2.1299 | Val   Acc: 0.7822


Training Epoch [15/20]:   0%|          | 0/782 [00:00<?, ?it/s]

Validating:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 0.0010 | Train Acc: 0.9997
Val   Loss: 2.1315 | Val   Acc: 0.7809
Early stopping triggered at epoch 15.
Training info: Dataset = Reconstructed (train_loader_recon), Classifier = Frozen layers (follows original classifier arch)
                  Dataset  CrossEntropy  Accuracy
0  Reconstructed Test Set      1.981306    0.7845
1       Original Test Set     58.336049    0.1122
